# DiD omitted-variable-bias sensitivity

This tutorial applies `DIDOVBSensitivity` to a balanced two-period panel. The estimator follows Wang, Sant'Anna, Chernozhukov, and Cinelli (2026): it cross-fits the treatment propensity and the untreated outcome change, then reports restricted omitted-variable-bias bounds, the robustness value (RV), and the extreme robustness value (XRV).

The sensitivity parameters describe a hypothetical omitted confounder. They do not prove that parallel trends holds and should be justified using the study context.

In [ ]:
import numpy as np
import pandas as pd

from diff_diff import DIDOVBSensitivity

## 1. Create a balanced two-period panel

The outcome is observed before and after treatment. The treatment is fixed at the unit level, while the covariates are measured before treatment.

In [ ]:
rng = np.random.default_rng(20260929)
n = 400
unit = np.arange(n)
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
propensity = 1 / (1 + np.exp(-(0.35 * x1 - 0.25 * x2)))
treated = rng.binomial(1, propensity)
y_pre = 0.5 * x1 - 0.25 * x2 + rng.normal(scale=1.0, size=n)
y_post = y_pre + 0.75 + 0.40 * treated + 0.20 * x1 + rng.normal(scale=1.0, size=n)

data = pd.DataFrame({
    'unit': np.tile(unit, 2),
    'time': np.repeat([0, 1], n),
    'treated': np.tile(treated, 2),
    'x1': np.tile(x1, 2),
    'x2': np.tile(x2, 2),
    'outcome': np.concatenate([y_pre, y_post]),
})
data.head()

## 2. Fit the sensitivity estimator

Set a seed when you want the cross-fitting partition to be reproducible. The default learners are a logistic propensity model and a linear untreated-trend model.

In [ ]:
result = DIDOVBSensitivity(n_folds=5, seed=17).fit(
    data,
    outcome='outcome',
    treatment='treated',
    time='time',
    unit='unit',
    covariates=['x1', 'x2'],
)
print(result.summary())

## 3. Translate sensitivity assumptions into ATT bounds

`trend_r2` is the fraction of residual untreated-trend variation explained by the omitted confounder. `selection_r2` is the corresponding fraction of residual treatment-odds variation. The bound uses the paper's DiD OVB scale decomposition.

In [ ]:
bounds = result.bounds(trend_r2=0.25, selection_r2=0.25)
print(bounds)

robustness = result.robustness_value(null_value=0.0, alpha=0.05)
print('RV:', robustness.rv)
print('XRV:', robustness.xrv)

The RV is the minimum equal strength of outcome-trend and treatment-selection confounding required to move the two-sided confidence interval to the null value. The XRV allows the omitted confounder to explain all remaining outcome-trend variation and therefore isolates the selection-strength requirement.